# 01 — Reproducible count baseline

This is a **simple count-based reference**, not a released EHRSHOT baseline reproduction and not the final model.

## Scope and safeguards

Features use structured codes recorded strictly before prediction. Vocabulary fitting uses training only; model selection and reporting use validation only. The test split is rejected by the reusable API.

In [ ]:
from pathlib import Path

from IPython.display import display

from temporal_ehr.config import resolve_path
from temporal_ehr.data.history import validate_requested_splits
from temporal_ehr.data.task_registry import load_task_registry
from temporal_ehr.features.count_baseline import DEFAULT_C_GRID, run_count_baseline

event_path = resolve_path(None, "EHRSHOT_DATA_PATH", must_be_file=True)
asset_root = resolve_path(None, "EHRSHOT_ASSET_ROOT", must_be_dir=True)
del asset_root
cohort_path = Path("outputs/private/cohorts/primary_panel.parquet")
if not cohort_path.is_file():
    command = (
        ".venv/bin/python -m temporal_ehr.cli build-cohort "
        '--asset-root "${EHRSHOT_ASSET_ROOT}" --tasks lab_anemia new_acutemi guo_los '
        "--selection-seed 20260814 --private-output outputs/private/cohorts/primary_panel.parquet "
        "--allow-private-output"
    )
    print(f"Missing governed cohort. Rebuild it exactly with:\n{command}")
    raise FileNotFoundError("primary_panel.parquet is required; no replacement cohort was created")
registry = load_task_registry(Path("configs/tasks.yaml"))
tasks = tuple(registry["primary_panel"])
modeling_splits = validate_requested_splits(("train", "validation"))

## Configuration

In [ ]:
configuration = {
    "name": "simple count-based reference",
    "tasks": tasks,
    "modeling_splits": modeling_splits,
    "test_split_inspected": False,
    "c_grid": DEFAULT_C_GRID,
    "tie_rule": "highest validation AUROC; smallest C wins an exact tie",
    "count_transform": "log1p on nonzero counts",
    "random_seed": 20260815,
}
display(configuration)

## Run the simple count-based reference

In [ ]:
results = run_count_baseline(
    event_path, cohort_path, tasks=tasks, c_grid=DEFAULT_C_GRID, random_seed=configuration["random_seed"]
)

## Validation estimates

Every metric below is a validation estimate. The prevalence-only reference uses the training prevalence as its fixed probability.

In [ ]:
display(
    [
        {
            "task": task,
            "selected_C": result["selected_C"],
            "candidate_results": result["candidate_results"],
            **result["metrics"],
        }
        for task, result in results["tasks"].items()
    ]
)

## Artifact manifest

In [ ]:
display(
    {
        "private_artifact_root": "outputs/private/count_baseline/",
        "aggregate_report": "reports/generated/count_baseline/validation_results.json",
        "source_notebooks_overwritten": False,
        "individual_predictions_displayed": False,
        "test_metrics_computed": False,
    }
)